# 3-практика · Регрессия (Regression): Бишкектеги батирлер

**Кыргыз Республикасынын Эсептөө палатасы · AI тренинги · 1-күн**

Теорияда сиз батирдин баасын **башыңызда** болжолдодуңуз. Азыр ошол эле сызыкты **код менен** тартабыз — анан аны аудитордун куралына айлантабыз.

⚠️ Маалымат синтетикалык, баалар шарттуу — калып (pattern) маанилүү.

## 0-кадам · Даярдык

In [ ]:
%pip install -q pandas scikit-learn matplotlib

## 1-кадам · 120 батир

Ар бир батир боюнча: аянты, району, кабаты, абалы жана **баасы (доллар)** — бул биздин болжолдоочу сан.

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(4)   # seed
n = 120

райондор = {"Борбор (Центр)": 1500, "Джал": 1150, "Асанбай": 1230, "Восток-5": 1080, "Аламедин-1": 1000}
абалдар = {"жаңы ремонт": 1.12, "орточо": 1.0, "эски": 0.88}

батирлер = pd.DataFrame({
    "аянт_м2": np.round(rng.uniform(32, 115, n), 0),
    "район": rng.choice(list(райондор), n),
    "кабат": rng.integers(1, 13, n),
    "абалы": rng.choice(list(абалдар), n, p=[0.3, 0.45, 0.25]),
})
негиз = батирлер["район"].map(райондор) * батирлер["абалы"].map(абалдар)
батирлер["баа_доллар"] = np.round((негиз * батирлер["аянт_м2"] * rng.normal(1, 0.055, n)) / 100, 0) * 100
батирлер["сатып_алуучу"] = "жеке адам"

# Бир "кызыктуу" сатып алуу... (азырынча сыр 🤫)
i = int(rng.integers(0, n))
батирлер.loc[i, ["аянт_м2", "район", "абалы", "сатып_алуучу"]] = [72, "Джал", "орточо", "мамлекеттик мекеме"]
батирлер.loc[i, "баа_доллар"] = round(1150 * 72 * 1.42 / 100) * 100

батирлер.head(8)

## 2-кадам · Тааныш сүрөт: аянт менен баа

Теориядагы оюндун так өзү — бирок эми 4 эмес, 120 чекит:

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(9, 5))
plt.scatter(батирлер["аянт_м2"], батирлер["баа_доллар"], color="#2C5F2D", alpha=0.6)
plt.xlabel("Аянты (м²)")
plt.ylabel("Баасы (доллар)")
plt.title("Бишкек батирлери: аянт менен баа")
plt.show()

## 3-кадам · 1-модель: сызыкты машина тартсын

Оюнда сызыкты башыңызда тарттыңыз. Азыр **сызыктуу регрессия (Linear Regression)** так ошону жасайт:

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error

модель_1 = LinearRegression()
модель_1.fit(батирлер[["аянт_м2"]], батирлер["баа_доллар"])
божомол_1 = модель_1.predict(батирлер[["аянт_м2"]])

plt.figure(figsize=(9, 5))
plt.scatter(батирлер["аянт_м2"], батирлер["баа_доллар"], color="#2C5F2D", alpha=0.5)
plt.plot(батирлер["аянт_м2"], божомол_1, color="#C9A227", linewidth=3)
plt.xlabel("Аянты (м²)"); plt.ylabel("Баасы (доллар)")
plt.title("Регрессия модели = ошол эле сызык")
plt.show()

mae_1 = mean_absolute_error(батирлер["баа_доллар"], божомол_1)
print(f"Орточо ката (MAE): ±${mae_1:,.0f}")

Модель орто эсеп менен **±$11 миңге** жаңылат. Эмнеге? Себеби ал **аянтты гана** билет — район менен абалын көрбөйт.

## 4-кадам · 2-модель: белгилерди кошобуз

Теориядагы убада: «көбүрөөк белги (features) → жакшыраак божомол». Текшерип көрөлү — район менен абалын кошолу:

In [ ]:
X_кеңири = pd.get_dummies(батирлер[["аянт_м2", "район", "абалы"]])

модель_2 = LinearRegression()
модель_2.fit(X_кеңири, батирлер["баа_доллар"])
божомол_2 = модель_2.predict(X_кеңири)

mae_2 = mean_absolute_error(батирлер["баа_доллар"], божомол_2)
print(f"1-модель (аянт гана):            ±${mae_1:,.0f}")
print(f"2-модель (аянт + район + абалы): ±${mae_2:,.0f}")
print(f"Ката {mae_1/mae_2:.1f} эсе азайды!")

## 5-кадам · 🖊 Өз батириңизди баалаңыз

Сандарды өзгөртүп, ар кандай батирлерди баалап көрүңүз:

In [ ]:
жаңы_батир = pd.DataFrame([{
    "аянт_м2": 65,              # 🖊 өзгөртүңүз
    "район": "Асанбай",         # 🖊 Борбор (Центр) / Джал / Асанбай / Восток-5 / Аламедин-1
    "абалы": "орточо",          # 🖊 жаңы ремонт / орточо / эски
}])
X_жаңы = pd.get_dummies(жаңы_батир).reindex(columns=X_кеңири.columns, fill_value=0)
print(f"Болжолдонгон баа: ${модель_2.predict(X_жаңы)[0]:,.0f}")

## 6-кадам · Регрессия — аудитордун чырагы 🔦

Эң кызыктуу жери. Ар бир батир үчүн эсептейли: **чыныгы баа − болжолдонгон баа**.

Айырма чоң болсо — баа «нормадан» алыс. Иреттеп, эң чоңун карайлы:

In [ ]:
батирлер["болжол"] = божомол_2.round(0)
батирлер["айырма"] = (батирлер["баа_доллар"] - батирлер["болжол"]).round(0)

батирлер.sort_values("айырма", ascending=False)[
    ["аянт_м2", "район", "абалы", "сатып_алуучу", "баа_доллар", "болжол", "айырма"]
].head(5)

## 😮 Таблицанын биринчи сабын караңыз

Бир батир болжолдон **$35 миңге кымбат** сатылып алынган — жана сатып алуучусу... **мамлекеттик мекеме**.

**Суроо:** Кайсы сатып алууну биринчи текшеребиз? 🙂

**Бул — регрессиянын аудиттеги күчү:** модель «норманы» үйрөнөт, ал эми нормадан четтегендер — текшерүүнүн даректери.

## → Эртеңки көпүрө

Бүгүн бизде **жооптор бар эле** (баалар, тобокелдик класстары). Ал эми эртең...
**эч ким жооп бербесе да** машина шектүү нерселерди кантип табарын көрөбүз — **Anomaly Detection**.

**1-күн бүттү. Азаматсыздар!** 🎉